# HW1 — Monte Carlo

**Course:** Reinforcement Learning — IE University Madrid

**Instructor:** Jaume Manero

**Author:** Elad Moshe

**Program:** MCSBT

**Date:** May 2026

---

### Notebooks Structure

1. **Activity 1 — Familiarisation with Gymnasium Environments**
2. Activity 2 — Monte Carlo on FrozenLake 4×4 and 8×8
3. Activity 3 — Monte Carlo on Volcano
4. Activity 4 — Monte Carlo on Taxi
5. Activity 5 — TD(0) on FrozenLake
6. Questions

---

> **AI Usage Disclaimer:** This notebook was developed with AI assistance (Claude, Anthropic). AI was used to improve documentation and markdown explanations, help structure the notebook, and assist with debugging implementation code. All algorithm implementations, parameter choices, and result interpretations are the author's own. As required by the course AI policy, care was taken to understand every critical update step.

---
<a id="activity-1"></a>
## Activity 1 — Familiarisation with Gymnasium Environments

For each environment we examine the **observation space** $S$, **action space** $A$, **reward function** $R$,
and whether the environment is continuous or discrete. Observation Space explanation is in the comments.

In [3]:
!pip install gymnasium

In [4]:
import sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

print(f"Python    : {sys.version.split()[0]}")
print(f"gymnasium : {gym.__version__}")
print(f"numpy     : {np.__version__}")

# ── CartPole-v1 ────────────────────────────────────────────────────────────────
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=42)

# Observation space: Box(4,) — 4 continuous values per state
#   obs[0] — Cart position        : horizontal position on the track  (range: -4.8 to 4.8 m)
#   obs[1] — Cart velocity        : speed of the cart left/right      (range: -inf to inf)
#   obs[2] — Pole angle           : angle of the pole from vertical   (range: ≈ -24° to +24°)
#   obs[3] — Pole angular velocity: rotation speed of the pole        (range: -inf to inf)

print("\n=== CartPole-v1 ===")
print(f"  Observation space : {env.observation_space}")
print(f"  Action space      : {env.action_space}")
print(f"  First observation : {obs}")
print()
print("  5 random steps:")
for i in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    print(f"    step {i+1} | action={action} | reward={reward:.1f} | terminated={terminated}")
    if terminated or truncated:
        obs, _ = env.reset()

env.close()

Python    : 3.13.13
gymnasium : 1.3.0
numpy     : 2.4.4

=== CartPole-v1 ===
  Observation space : Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
  Action space      : Discrete(2)
  First observation : [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]

  5 random steps:
    step 1 | action=1 | reward=1.0 | terminated=False
    step 2 | action=0 | reward=1.0 | terminated=False
    step 3 | action=1 | reward=1.0 | terminated=False
    step 4 | action=1 | reward=1.0 | terminated=False
    step 5 | action=0 | reward=1.0 | terminated=False


In [5]:
# ── CartPole-v1 visual rendering ──────────────────────────────────────────────
env = gym.make("CartPole-v1", render_mode="human")
obs, _ = env.reset(seed=42)

for i in range(50):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        obs, _ = env.reset()

env.close()

In [6]:
# ── Taxi-v4 ───────────────────────────────────────────────────────────────────
env = gym.make("Taxi-v4")
obs, _ = env.reset(seed=42)

# Observation space: Discrete(500) — single integer encoding the full state
#   Decoded as: taxi_row (0-4) × taxi_col (0-4) × passenger_loc (0-4) × destination (0-3)
#   taxi_row      : row of the taxi on the 5×5 grid      (0=top, 4=bottom)
#   taxi_col      : column of the taxi on the 5×5 grid   (0=left, 4=right)
#   passenger_loc : where the passenger currently is     (0=Red, 1=Green, 2=Yellow, 3=Blue, 4=In taxi)
#   destination   : drop-off location for the passenger  (0=Red, 1=Green, 2=Yellow, 3=Blue)

print("=== Taxi-v4 ===")
print(f"  Observation space : {env.observation_space}  ({env.observation_space.n} discrete states)")
print(f"    Encodes: taxi_row(0-4) x taxi_col(0-4) x passenger(0-4) x destination(0-3)")
print(f"    = 5 x 5 x 5 x 4 = 500 states")
print(f"  Action space : {env.action_space}")
print(f"    0=South  1=North  2=East  3=West  4=Pick-up  5=Drop-off")
print(f"  Type   : fully discrete")
print(f"  Reward : +20 successful drop-off | -1 per step | -10 illegal action")
print()

# ── Decode helper ─────────────────────────────────────────────────────────────
PASS_LABELS = {0: "Red", 1: "Green", 2: "Yellow", 3: "Blue", 4: "In taxi"}
DEST_LABELS = {0: "Red", 1: "Green", 2: "Yellow", 3: "Blue"}
ACTION_LABELS = {0: "South", 1: "North", 2: "East", 3: "West", 4: "Pick-up", 5: "Drop-off"}

def decode_state(state):
    taxi_row, taxi_col, pass_loc, dest_idx = env.unwrapped.decode(state)
    return taxi_row, taxi_col, pass_loc, dest_idx

def print_state(step, state, action, reward):
    taxi_row, taxi_col, pass_loc, dest_idx = decode_state(state)
    print(f"  Step {step:>3} | state={state:>3} | action={ACTION_LABELS[action]:<8}"
          f"| reward={reward:>4} "
          f"| taxi=({taxi_row},{taxi_col}) "
          f"| passenger={PASS_LABELS[pass_loc]:<8}"
          f"| dest={DEST_LABELS[dest_idx]}")

# ── Live loop for 10 seconds ──────────────────────────────────────────────────
print("Live run (10 seconds, random policy):")
print("-" * 90)

obs, _ = env.reset(seed=42)
start = time.time()
step = 0

while time.time() - start < 10:
    action = env.action_space.sample()
    new_obs, reward, terminated, truncated, _ = env.step(action)
    print_state(step, new_obs, action, reward)
    time.sleep(0.1)
    step += 1
    if terminated or truncated:
        print(f"  --- episode done (reward={reward}) | resetting ---")
        obs, _ = env.reset()
    else:
        obs = new_obs

env.close()
print("-" * 90)
print(f"Total steps in 10s: {step}")

=== Taxi-v4 ===
  Observation space : Discrete(500)  (500 discrete states)
    Encodes: taxi_row(0-4) x taxi_col(0-4) x passenger(0-4) x destination(0-3)
    = 5 x 5 x 5 x 4 = 500 states
  Action space : Discrete(6)
    0=South  1=North  2=East  3=West  4=Pick-up  5=Drop-off
  Type   : fully discrete
  Reward : +20 successful drop-off | -1 per step | -10 illegal action

Live run (10 seconds, random policy):
------------------------------------------------------------------------------------------
  Step   0 | state=386 | action=Pick-up | reward= -10 | taxi=(3,4) | passenger=Green   | dest=Yellow
  Step   1 | state=386 | action=Pick-up | reward= -10 | taxi=(3,4) | passenger=Green   | dest=Yellow
  Step   2 | state=366 | action=West    | reward=  -1 | taxi=(3,3) | passenger=Green   | dest=Yellow
  Step   3 | state=266 | action=North   | reward=  -1 | taxi=(2,3) | passenger=Green   | dest=Yellow
  Step   4 | state=246 | action=West    | reward=  -1 | taxi=(2,2) | passenger=Green   | dest=

In [7]:
# ── Taxi-v4 visual demo ───────────────────────────────────────────────────────
env = gym.make("Taxi-v4", render_mode="human")
obs, _ = env.reset(seed=42)

for i in range(15):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    time.sleep(0.05)
    if terminated or truncated:
        obs, _ = env.reset()

env.close()

In [8]:
# ── LunarLander-v3 visual demo ───────────────────────────────────────────────
# If not installed: pip install gymnasium[box2d]

# Observation space: Box(8,) — 8 continuous values per state
#   obs[0] — x position        : horizontal position of the lander            (range: ~ -1.5 to 1.5)
#   obs[1] — y position        : vertical position of the lander              (range: ~ -1.5 to 1.5)
#   obs[2] — x velocity        : horizontal speed                             (range: -inf to inf)
#   obs[3] — y velocity        : vertical speed                               (range: -inf to inf)
#   obs[4] — angle             : tilt angle of the lander body                (range: -π to π)
#   obs[5] — angular velocity  : rotation speed of the lander                 (range: -inf to inf)
#   obs[6] — left leg contact  : whether the left leg is touching the ground  (0 or 1)
#   obs[7] — right leg contact : whether the right leg is touching the ground (0 or 1)

env = gym.make("LunarLander-v3", render_mode="human")
obs, _ = env.reset(seed=42)

for i in range(250):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    time.sleep(0.03)
    if terminated or truncated:
        obs, _ = env.reset()

env.close()

In [9]:
# ── FrozenLake-v1 ─────────────────────────────────────────────────────────────
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
obs, _ = env.reset(seed=42)

# Observation space: Discrete(16) — single integer representing the agent's cell on the 4×4 grid
#   state = row * 4 + col  (both 0-indexed from top-left)
#   e.g. state  0 = top-left corner     (S — Start)
#        state  5 = row 1, col 1        (H — Hole, terminal)
#        state 15 = bottom-right corner (G — Goal, terminal)
#   Cell types: S=Start  F=Frozen (safe)  H=Hole (terminal, reward=0)  G=Goal (terminal, reward=1)

print("=== FrozenLake-v1 (4x4, non-slippery) ===")
print(f"  Observation space : {env.observation_space}  ({env.observation_space.n} states on 4x4 grid)")
print(f"  Action space : {env.action_space}  (0=Left  1=Down  2=Right  3=Up)")
print(f"  Type   : fully discrete")
print(f"  Reward : +1 reaching goal G | 0 otherwise (including falling into a hole)")
print(f"  Terminal states : H (holes) and G (goal)")
print(f"  Sample state : {obs}")
print()
print("  Grid layout (S=Start  F=Frozen  H=Hole  G=Goal):")
for row in env.unwrapped.desc:
    print("   ", " ".join(c.decode() for c in row))

print()
print("  5 random steps:")
for i in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    print(f"    step {i+1} | action={action} | reward={reward} | terminated={terminated} | new state={obs}")

env.close()

=== FrozenLake-v1 (4x4, non-slippery) ===
  Observation space : Discrete(16)  (16 states on 4x4 grid)
  Action space : Discrete(4)  (0=Left  1=Down  2=Right  3=Up)
  Type   : fully discrete
  Reward : +1 reaching goal G | 0 otherwise (including falling into a hole)
  Terminal states : H (holes) and G (goal)
  Sample state : 0

  Grid layout (S=Start  F=Frozen  H=Hole  G=Goal):
    S F F F
    F H F H
    F F F H
    H F F G

  5 random steps:
    step 1 | action=2 | reward=0 | terminated=False | new state=1
    step 2 | action=2 | reward=0 | terminated=False | new state=2
    step 3 | action=0 | reward=0 | terminated=False | new state=1
    step 4 | action=0 | reward=0 | terminated=False | new state=0
    step 5 | action=0 | reward=0 | terminated=False | new state=0


In [ ]:
# ── FrozenLake-v1 visual demo ────────────────────────────────────────────────
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="human")
obs, _ = env.reset(seed=42)

for i in range(50):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    time.sleep(0.05)
    if terminated or truncated:
        obs, _ = env.reset()

env.close()

: 

### Environment Comparison Summary

| Feature | CartPole-v1 | Taxi-v3 | LunarLander | FrozenLake-v1 |
|---------|-------------|---------|-------------|---------------|
| **Observation type** | Continuous Box | Discrete (500) | Continuous Box | Discrete (16) |
| **Action type** | Discrete (2) | Discrete (6) | Discrete (4) | Discrete (4) |
| **State space size** | Infinite | 500 | Infinite | 16 |
| **Reward on fail** | Episode ends | −1 / −10 | −100 crash | 0 (no penalty) |
| **Stochastic?** | No | No | No | Optional (`is_slippery`) |
| **Tabular RL suitable?** | No | Yes | No | Yes |

**Key differences:**
- **CartPole and LunarLander** have continuous observation spaces — tabular methods (MC, TD) cannot be applied directly. 
- **Taxi and FrozenLake** are fully discrete — a simple Q-table covers every state, making them potentially suitable for tabular RL methods.
- **Taxi** is the hardest discrete environment: 500 states, sparse reward (+20 on delivery), random starting configurations, and a sequential two-part task (pick-up then drop-off).
- **FrozenLake** is the simplest discrete environment: only 16 states, a fixed start, and a single goal. Adding `is_slippery=True` introduces stochasticity.